# Lecture 07 Policy

## 模块一：策略学习核心术语与符号定义

策略学习就是建立从“感知”到“动作”的映射。

*   **状态 (State, $s_t$)**: 环境的真实物理状态（通常在真实世界中不可见）。
*   **观测 (Observation, $o_t$)**: 外在传感器（如相机）和内在传感器（机械臂角度）获取到的信息。
*   **动作 (Action, $a_t$)**: 机器人执行的控制指令。
*   **策略 (Policy, $\pi$)**: 决策模型，通常用神经网络参数化为 $\theta$。
    *   **全观测 Fully observed 策略**: $\pi_\theta(a_t|s_t)$ 机器人一般做不到。
    *   **部分观测策略**: $\pi_\theta(a_t|o_t)$
*   **马尔可夫性质 (Markov Property)**: 下一步的状态只取决于当前状态和动作，与历史无关：$p(s_{t+1}|s_t, a_t)$ ，这其实就是 World Model，描述了世界的动力学 Dynamics。但真实情况中 $s_t$ 不一定考虑到了如精神状态等因素，所以不一定满足马尔可夫性。


---

## 模块二：模仿学习 (Imitation Learning, IL)

当存在最优策略或专家示教数据时，可以将策略学习转化为监督学习问题，即**行为克隆 (Behavioral Cloning, BC)**。

### 1. 核心思想
给定专家数据集 $\mathcal{D} = \{(o_i, a_i)\}$，通过监督学习直接拟合：$o_t \xrightarrow{\text{NN}} a_t$。
*   *数据来源*：人类遥操作 Teleoperation（如 Tesla、ALOHA 系统（一个人手拿着的 master arm，一个被遥控工作的 slave arm）、最优计算策略或教师策略。
*   当动作是离散的空间，本质和 Image classification 没有区别。是一种 supervised learning。
*   当动作是连续的空间，本质也是要 max log likelihood。

### 2. 致命缺陷：分布偏移 (Distributional Drift)
这是模仿学习的核心痛点（如 ALVINN 自动驾驶案例）。
*   **原因**：训练时，数据服从专家分布 $p_{\text{data}}(o_t)$；但在部署时，策略的微小误差会导致轨迹偏离，进入未见过的状态，即测试分布 $p_{\pi_\theta}(o_t) \neq p_{\text{data}}(o_t)$，导致误差累积（Error Compounding）并最终失败。

### 3. 解决方案 A：从数据层面解决 (收集更多数据)
**DAgger (Dataset Aggregation) 算法**
*   **目标**：强制让 $p_{\text{data}}(o_t) = p_{\pi_\theta}(o_t)$。
*   **流程**：
    1.  用人类数据 $\mathcal{D}$ 训练初始策略 $\pi_\theta$。
    2.  运行 $\pi_\theta$ 收集新的状态轨迹集 $\mathcal{D}_\pi$（让机器自己去跑，暴露出偏移的状态）。
    3.  **难点**：让人类专家对 $\mathcal{D}_\pi$ 中的每一个状态标注正确的动作 $a_t$。
    4.  合并数据集 $\mathcal{D} \leftarrow \mathcal{D} \cup \mathcal{D}_\pi$，重复步骤 1。
*   **问题**：让人类对机器犯错的中间状态进行补救标注是非常不自然的（Unnatural）。
*   现代 Dagger：让机器自己先做，做的快出问题了，人遥操接管，作为新增数据。这种时实接管，存在主臂与从臂在接管的一瞬间不一定处于同一状态的问题。可能的解决方案是利用Cartesian Space而非Configuration Space。
    | 维度 | Configuration Space（构型空间 / 关节空间） | Cartesian Space（笛卡尔空间 / 操作空间） |
    |:---|:---|:---|
    | **状态表示** | 关节角度/位置向量 `q = [θ₁, θ₂, ..., θₙ] ∈ ℝⁿ` | 末端执行器位姿 `X = [x, y, z, roll, pitch, yaw]` 或 `[p, q]` ∈ ℝ⁶ |
    | **控制对象** | 直接下发给电机/关节驱动器 | 作用于机器人末端在三维世界中的运动 |
    | **人类直觉** | 反直觉（“关节3转0.15rad”） | 符合直觉（“夹爪向左移5cm，俯仰转10°”） |
    | **映射关系** | 通过 **Forward Kinematics (FK)** → 末端位姿 | 通过 **Inverse Kinematics (IK)** → 关节指令 |
    | **典型场景** | 底层伺服、轨迹规划、动力学控制 | 遥操指令、任务级规划、人机交互 |

    **为什么 Cartesian + ΔR/ΔT + IK 能缓解/解决该问题？**  
    接管瞬间，系统**不依赖主控臂的绝对关节状态**，而是：
    - 实时读取从臂当前末端位姿 `X_current`（通过 FK 高频刷新）
    - 人类只需输入**相对增量** `ΔX = [ΔT, ΔR]`
    - 指令基准自动对齐到从臂真实状态，消除绝对跳变
    
    **为什么 Off-policy DAgger 存在“重复修正”（Repeated Correction）问题？**
    1. 状态访问分布偏移（Covariate Shift）
    - **On-policy DAgger**：每次训练的损失期望严格对齐当前策略的访问分布  
      $\mathcal{L}_{\text{on}} = \mathbb{E}_{s \sim d_{\pi_k}(s)}[\ell(\pi_\theta(s), a^*)]$
    - **Off-policy DAgger**：数据集 $D = \bigcup_{i=0}^{k-1} \{(s_t^i, a_t^{i*})\}$ 包含过去所有策略的观测。训练时通常直接做经验风险最小化：  
      $\mathcal{L}_{\text{off}} = \frac{1}{|D|}\sum_{(s,a^*)\in D} \ell(\pi_\theta(s), a^*)$
    - **问题**：$\pi_k$ 已经学会避开某些历史状态（如奇异点、易碰撞区域），但 $D$ 中仍大量存在这些状态。策略被迫**反复学习它已经不再访问的状态**，形成“刻舟求剑”式的重复修正。

    2. 梯度方向冲突与振荡
    - 不同历史策略 $\pi_i, \pi_j$ 在同一物理场景可能因轨迹不同而触发不同的专家干预。
    - 混合数据集的梯度是多个历史分布梯度的线性叠加：  
      $\nabla_\theta \mathcal{L}_{\text{off}} \approx \sum_i w_i \mathbb{E}_{s \sim d_{\pi_i}}[\nabla_\theta \ell]$
    - 当 $\pi_k$ 的实际轨迹分布 $d_{\pi_k}$ 与历史分布差异较大时，梯度会**在策略参数空间中来回拉扯**，表现为：修正状态A → 破坏状态B → 再次修正A，收敛缓慢甚至性能回退。


### 4. 解决方案 B：从模型层面解决 (提高策略拟合能力)
如果不增加数据，就需要模型极其完美地拟合专家，不产生初始偏差。传统 BC 失败的两个本质原因及对策：
*   **原因 1：非马尔可夫行为 (Non-Markovian Behavior)**
    *   *现象*：专家决策依赖历史信息，而模型只看当前帧。
    *   *对策*：引入历史帧。为了避免权重爆炸，使用共享权重的 **RNN / LSTM** 来提取历史状态。
    *   *隐患*：因果混淆 (Causal Confusion)——模型可能学到了虚假的因果关系（例如看到刹车灯亮才刹车，而不是看到行人刹车）。
*   **原因 2：多模态行为 (Multimodal Behavior)**
    *   *现象*：面对同一棵树，专家有时从左绕，有时从右绕。均方误差 (MSE) 会让模型取平均值（直接撞树）。
    *   *对策*：
        1. 输出高斯混合模型 (Mixture of Gaussians)。
        2. 离散化自回归模型 (Autoregressive discretization)：比如把走的方向分成左中右三个方向，然后变成一个classification问题，如果中间有障碍，那根据gt，模型最后学到的概率分布将会是0.5 0 0.5。
        3. 隐变量模型 (Latent variable models, 如 VAE)。
        4. **Diffusion Policy (扩散策略)**：通过去噪扩散过程生成动作，是目前具身智能（如机械臂操作）处理多模态分布的 SOTA 方法。

**L2 Loss 为什么隐含高斯/单峰假设？**  
L2 Loss 最小化均方误差，等价于在“条件高斯分布”假设下做极大似然估计；它天然输出条件期望 $\mathbb{E}[a|s]$，对多峰数据会做“模式平均（Mode Averaging）”，导致策略输出模糊、保守甚至危险的动作。

假设真实值 $y$ 与预测值 $\hat{y}$ 之间的误差 $\epsilon = y - \hat{y}$ 服从 **均值为 0、方差为 $\sigma^2$ 的高斯分布**：

$$
p(\epsilon) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{\epsilon^2}{2\sigma^2}\right)
$$

即给定输入 $x$ 时，输出 $y$ 的条件概率为：

$$
p(y \mid x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y - \hat{y})^2}{2\sigma^2}\right)
$$

对单个样本取**负对数似然**：

$$
-\log p(y \mid x) = \frac{1}{2\sigma^2} (y - \hat{y})^2 + \frac{1}{2}\log(2\pi\sigma^2)
$$

其中 $\frac{1}{2\sigma^2} > 0$ 为常数，$\frac{1}{2}\log(2\pi\sigma^2)$ 为与 $\hat{y}$ 无关的常数。

因此：
$$
\arg\min_{\hat{y}} \left[ -\log p(y \mid x) \right]
= \arg\min_{\hat{y}} \left[ \frac{1}{2\sigma^2} (y - \hat{y})^2 + \text{constant} \right]
= \arg\min_{\hat{y}} (y - \hat{y})^2
$$
即最小化负对数似然等价于最小化平方误差 $(y - \hat{y})^2$。

> 💡 关键点：L2 不关心数据有几个峰值，它只关心“平方误差最小”。当分布是多峰时，数学期望会落在**低概率区域**（两峰之间），策略被迫输出“折中动作”。


---

## 模块三：强化学习 (Reinforcement Learning, RL)

当没有专家示教，或者不知道什么是“正确”动作时，只能依靠**试错 (Trial and Error)**。
*   **与监督学习的区别**：数据不是独立同分布的（i.i.d.，当前动作影响未来状态）；没有 Ground Truth，只有成功/失败或延迟的奖励信号。

### 1. 数学形式化：马尔可夫决策过程 (MDP)
一个 MDP 定义为元组 $\mathcal{M} = \{\mathcal{S}, \mathcal{A}, \mathcal{T}, r\}$：
*   $\mathcal{S}$: 状态空间
*   $\mathcal{A}$: 动作空间
*   $\mathcal{T}$: 转移概率张量 $p(s_{t+1}|s_t, a_t)$
*   $r$: 奖励函数 $r(s_t, a_t) \to \mathbb{R}$
*   *(若是部分观测，则扩展为 POMDP，加入观测空间 $\mathcal{O}$ 和发射概率 $\mathcal{E}: p(o_t|s_t)$)*。

### 2. 强化学习的优化目标
定义一条交互轨迹 $\tau = (s_1, a_1, \dots, s_T, a_T)$，其发生的概率为：
$$p_\theta(\tau) = p(s_1) \prod_{t=1}^T \pi_\theta(a_t|s_t) p(s_{t+1}|s_t, a_t)$$

强化学习的目标是最大化**期望累计奖励**：
$$\theta^* = \arg\max_\theta J(\theta) = \arg\max_\theta \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[ \sum_{t=1}^T r(s_t, a_t) \right]$$
*   *关键理解*：奖励函数 $r(x)$ 本身可能是不平滑的（如 0/1 阶跃信号），但**期望 $\mathbb{E}[r(x)]$ 关于参数 $\theta$ 是平滑的**，因此可以通过梯度下降优化。

在 theta 下对全部 tao（轨迹）空间的 reward 的数学期望：
$$
J(\theta) = E_{\tau \sim p_\theta(\tau)}[\underbrace{r(\tau)}_{\sum_{t=1}^T r(\mathbf{s}_t, \mathbf{a}_t)}] = \int p_\theta(\tau)r(\tau)d\tau
$$
对上式求导，通过如下推导可以变成一个新的数学期望：
$$
\nabla_\theta J(\theta) = \int \underline{\nabla_\theta p_\theta(\tau)}r(\tau)d\tau = \int \underline{p_\theta(\tau)\nabla_\theta \log p_\theta(\tau)}r(\tau)d\tau = E_{\tau \sim p_\theta(\tau)}[\nabla_\theta \log p_\theta(\tau)r(\tau)]
$$
通过把对数部分里面与theta无关的去掉，可以简化成下式：
$$
\nabla_\theta J(\theta) = E_{\tau \sim p_\theta(\tau)} \left[ \left( \sum_{t=1}^T \nabla_\theta \log \pi_\theta(\mathbf{a}_t|\mathbf{s}_t) \right) \left( \sum_{t=1}^T r(\mathbf{s}_t, \mathbf{a}_t) \right) \right]
$$
这个新的数学期望，依然不好直接算，用蒙特卡洛采样方法进行计算，得到策略梯度的无偏估计为：
$$
\nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \left( \sum_{t=1}^T \nabla_\theta \log \pi_\theta(\mathbf{a}_{i,t}|\mathbf{s}_{i,t}) \right) \left( \sum_{t=1}^T r(\mathbf{s}_{i,t}, \mathbf{a}_{i,t}) \right)
$$

*   **致命痛点：高方差 (High Variance)**。由于轨迹空间极其庞大，用有限的采样 $N$ 来评估整条轨迹的期望回报，会产生巨大的噪声，导致训练极不稳定。**本节课后续的所有内容，都是为了降低方差 (Reducing Variance)**。



#### 为什么说 RL 的精髓在于 $r$ 是什么都可以，主要靠 $p_\theta$？

在传统的深度学习（比如图像分类、行为克隆）中，我们的损失函数（Loss）必须是**可导的（Differentiable）**，比如均方误差（MSE）或交叉熵（Cross Entropy）。因为我们要靠链式法则把梯度一路反向传播（Backpropagation）回神经网络。

但在强化学习（尤其是具身智能）中，奖励函数 $r$ 往往是**不可导的、甚至是非常离散和不平滑的黑盒**。
*   **例子**：让机器人学走路，奖励函数可能是：机器人没摔倒给 `+1`，摔倒了给 `-1`。这是一个阶跃信号，根本没有梯度！如果对这个 $r$ 求导，梯度要么是 0，要么是无穷大。

**那么策略梯度是怎么绕过这个致命问题的呢？** 靠的就是 $p_\theta$（轨迹分布概率）和**对数导数技巧（Log-derivative trick）**。

我们看策略梯度的核心公式推导：
我们的目标是最大化期望奖励：
$$ J(\theta) = \mathbb{E}_{\tau \sim p_\theta(\tau)} [r(\tau)] = \int p_\theta(\tau) r(\tau) d\tau $$

对目标求导，奇迹发生了：
$$ \nabla_\theta J(\theta) = \int \nabla_\theta p_\theta(\tau) r(\tau) d\tau = \int p_\theta(\tau) \nabla_\theta \log p_\theta(\tau) r(\tau) d\tau = \mathbb{E}_{\tau \sim p_\theta(\tau)} [\nabla_\theta \log p_\theta(\tau) \cdot r(\tau)] $$

**仔细看最后这个公式，这是整个 RL 的灵魂：**
1. 梯度算子 $\nabla_\theta$ **完全越过了** $r(\tau)$，直接作用在了 $\log p_\theta(\tau)$（即我们的神经网络策略 $\pi_\theta$）上！
2. 在求梯度的过程中，**$r(\tau)$ 仅仅扮演了一个标量权重（Scalar Weight）的角色**。它不需要被求导！

因为我们**只对神经网络产生的概率分布 $p_\theta$ 求导**（神经网络是天然平滑且可导的），所以奖励 $r$ 可以是**任何东西**！它可以是 0 或 1 的离散值，可以是代码里的一个 `if-else` 黑盒逻辑，甚至可以是人类主观给出的评分。只要你能给出一个数字 $r$，策略梯度就能用这个数字去放大或缩小那些产生这个 $r$ 的动作概率。这就极大地拓宽了 RL 的应用场景。

#### 为什么说蒙特卡罗（Monte Carlo）方法是“无偏的”（Unbiased），但是“方差巨大”（High Variance）？

要理解这个，我们先明确蒙特卡罗（MC）在 RL 里是怎么做的：**蒙特卡罗就是“把一局游戏玩到底，然后把一路上的奖励直接加起来”**，作为当前状态的回报估计。

##### 1. 为什么是“无偏的”（Unbiased）？
在统计学中，“无偏”的意思是：**虽然你每次抛硬币的结果都不一样，但你抛一万次的平均值，完美等于它真实的数学期望。**

在 RL 中，状态价值 $V(s)$ 的严谨数学定义，就是“从状态 $s$ 开始，未来真实能拿到的累计奖励的**期望值**”。
蒙特卡罗方法不借助任何其他神经网络去“猜”未来的奖励，而是老老实实地让机器人在物理引擎里把未来走完，拿到真实的 $\sum r$。
因为它是来自真实环境的**第一手观测数据**，它没有夹杂任何模型的错误假设，所以它的期望值**完美等于真实的回报期望**，这就叫“无偏”。

##### 2. 为什么“方差巨大”（High Variance）？
“方差大”的意思是：**每一次采样得到的结果，上下波动极其剧烈。**

想象一个抛硬币赌博游戏，赢了给你 1000 万，输了你赔 1000 万。这个游戏的期望（无偏估计）是 0，但是它的方差大得吓人。

在 RL 的蒙特卡罗采样中，方差巨大的原因在于**“蝴蝶效应”**和**极高的维度**：
*   **动作的随机性**：假设一条轨迹要走 1000 步（$T=1000$）。在第 10 步的时候，机器人的策略网络输出了一个概率分布，由于采样的随机性，它手抖了一下，往左多偏了 1 厘米。
*   **环境的随机性**：就因为这 1 厘米，导致它错过了抓取物体的最佳角度，后面 990 步全盘崩溃，最后总奖励是 `-100`。
*   而在下一次采样时，同样是在第 10 步，它随机采样到了一个完美的动作，顺利完成任务，总奖励是 `+100`。

**同样的初始状态，仅仅因为漫长的轨迹中某些微小的随机变化，导致蒙特卡罗算出来的总奖励（Reward-to-go）一会儿是 -100，一会儿是 +100。**

这时候你用这个剧烈波动的数字去更新神经网络：
$$ \theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta \cdot (\text{一会儿是+100，一会儿是-100}) $$
神经网络就会被这股“东风”和“西风”吹得晕头转向，梯度更新的方向左右横跳，极难收敛。这就是老师说的**高方差（High Variance）灾难**。

#### 为什么说 RL 绝不是仅仅给 IL 加了个权重
##### 第一步：为什么 IL (最大似然估计) 是下面那个公式？

在模仿学习（Imitation Learning / 行为克隆）中，我们有一个**专家数据集**，里面记录了专家在遇到状态 $\mathbf{s}_{i,t}$ 时，所做出的完美动作 $\mathbf{a}_{i,t}^*$。

我们的目标是训练一个策略网络 $\pi_\theta$，让它在看到同样的 $\mathbf{s}_{i,t}$ 时，输出专家动作 $\mathbf{a}_{i,t}^*$ 的概率越大越好。

在数学上，这叫**最大似然估计 (Maximum Likelihood Estimation, MLE)**。我们要最大化专家动作的对数概率：
$$ J_{\text{ML}}(\theta) = \frac{1}{N} \sum_{i=1}^N \sum_{t=1}^T \log \pi_\theta(\mathbf{a}_{i,t}^* | \mathbf{s}_{i,t}) $$

对目标函数求导，得到的就是 PPT 下方的公式：
$$ \nabla_\theta J_{\text{ML}}(\theta) \approx \frac{1}{N} \sum_{i=1}^N \left( \sum_{t=1}^T \nabla_\theta \log \pi_\theta(\mathbf{a}_{i,t}^* | \mathbf{s}_{i,t}) \right) $$

**结论**：这个梯度的物理意义是——**无论如何，无脑拉高这些特定动作 $\mathbf{a}^*$ 的输出概率**。这个梯度是**不为 0 的**（除非你的网络已经 100% 预测对了，但这在连续空间是不可能的），它会一直拽着网络参数往专家的方向走。

---

##### 第二步：思想实验 —— 如果 RL 中所有 $r=1$，为什么梯度是 0？

现在我们来看上面的 Policy Gradient 公式。假设这是一个极其无聊的环境，无论机器人做什么动作，环境给的奖励永远是 $r=1$。

把 $r=1$ 代入 PG 公式，后面那个求和项 $\sum r$ 就变成了一个常数 $T$（轨迹长度）。此时 PG 的梯度期望变成了：
$$ \nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(\tau) \cdot T \right] = T \cdot \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(\tau) \right] $$

重点来了，**在数学上，任何概率分布对数梯度的期望，严格等于 0！**
推导非常简单（利用 $\nabla \log f(x) = \frac{\nabla f(x)}{f(x)}$）：
$$ \mathbb{E}_{x \sim p_\theta} [\nabla_\theta \log p_\theta(x)] = \int p_\theta(x) \frac{\nabla_\theta p_\theta(x)}{p_\theta(x)} dx = \int \nabla_\theta p_\theta(x) dx = \nabla_\theta \int p_\theta(x) dx = \nabla_\theta (1) = 0 $$

**物理直觉解释**：
如果所有动作给的奖励都是 1，意味着**所有的动作一样好**。既然目前策略做任何事都能拿到一样的最高分，那就**没有优化的方向了**，网络参数不需要也不应该发生任何改变，因此梯度必须是 0。

---

##### 第三步：为什么同样没有 $r$，PG 算出来是 0，IL 算出来不是 0？（你的核心疑问）

**数据来源（Data Distribution）不同！**

仔细看这两个公式中的动作 $\mathbf{a}$：

1.  **在 Policy Gradient (RL) 中**：
    数据是用**当前的策略自己**跑出来的，即 $\mathbf{a} \sim \pi_\theta$。
    因为是你自己采样出来的动作，那些大概率的动作被采样到的次数多，小概率的动作被采样到的次数少。如果你不对它们进行奖励的区分（即 $r$ 全等于常数），网络自身的概率加权和就抵消成了 0。**（自己生成数据，如果没有外部评分，就不产生更新）**。

2.  **在 Maximum Likelihood (IL) 中**：
    数据是**外部专家**给的，即 $\mathbf{a} \sim \pi_{\text{expert}}$，而不是 $\pi_\theta$。
    此时，期望是在专家数据分布下求的：$\mathbb{E}_{\mathbf{a} \sim \pi_{\text{expert}}} [\nabla_\theta \log \pi_\theta(\mathbf{a}|\mathbf{s})]$。
    由于 $\pi_{\text{expert}}$ 和你当前的 $\pi_\theta$ 是**完全不同的两个分布**，上述那个等于 0 的数学恒等式就不成立了！
    **（外部强塞给你的数据，你的网络必须改变参数去迎合它，所以梯度不为 0）**。

## 模块四：降低方差的三大数学武器

如何在不引入偏差（Unbiased）的前提下降低策略梯度的方差？课程给出了三个循序渐进的技巧。

### 武器 1：利用因果性 (Causality) —— Reward-to-go
*   **物理直觉**：当前的动作 $a_t$ 只能影响未来的奖励，不能影响过去的奖励。
*   **数学修正**：将对整条轨迹的总奖励求和，替换为从当前时刻 $t$ 开始的**“未来回报 (Reward-to-go)”** $\hat{Q}_{i,t}$。
    $$ \nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=1}^T \nabla_\theta \log \pi_\theta(a_{i,t}|s_{i,t}) \underbrace{\left( \sum_{t'=t}^T r(s_{i,t'}, a_{i,t'}) \right)}_{\text{Reward-to-go } \hat{Q}_{i,t}} $$

### 武器 2：引入基线 (Baselines)
*   **数学证明**：从回报中减去一个与当前动作无关的基线 $b(s_t)$，**不会改变梯度的期望（即无偏）**。
    $$ \mathbb{E}_{a_t \sim \pi_\theta}[\nabla_\theta \log \pi_\theta(a_t|s_t) b(s_t)] = \nabla_\theta \int \pi_\theta(a_t|s_t) b(s_t) da_t = \nabla_\theta (1 \cdot b(s_t)) = 0 $$
*   **最优基线**：理论上存在一个最小化方差的最优基线（用梯度幅度的平方加权的期望奖励），但极难计算。
*   **实用基线**：通常使用状态价值函数 $V^\pi(s_t)$ 作为基线。

### 武器 3：折扣因子 (Discount Factor, $\gamma$)
*   在无限期 (Infinite-horizon) 任务中，奖励之和会发散。引入 $\gamma \in[0, 1)$。
*   **本质作用 = 降低方差**：$\gamma$ 使得越久远的未来奖励权重越小（因为久远的未来受当前动作的影响更小，带来的主要是噪声）。$\gamma$ 越小，考虑的视野越短，方差越低，但会引入一定偏差。
*   **修正后的折扣回报 (Discounted Return)**：
    $$ G_t = \sum_{t'=t}^\infty \gamma^{t'-t} r_{t'} $$

## 模块五：Actor-Critic 算法架构 (The Actor-Critic Architecture)

即便用了 Reward-to-go，单次采样的 Monte Carlo 估计依然充满随机性。如果能训练一个神经网络来“直接预测”期望回报，方差会大幅下降。这就引出了 Actor-Critic。

#### 1. 核心概念引入：Q函数与V函数
*   **Q-function (状态-动作价值)** $Q^\pi(s_t, a_t)$：在状态 $s_t$ 执行动作 $a_t$ 后，未来奖励的期望。
*   **Value function (状态价值)** $V^\pi(s_t)$：在状态 $s_t$ 下，未来的期望奖励。
*   **Advantage function (优势函数)** $A^\pi(s_t, a_t) = Q^\pi(s_t, a_t) - V^\pi(s_t)$：衡量当前动作 $a_t$ 比平均动作“好多少”。

#### 2. Actor-Critic 的数学推导
根据全期望公式（Law of total expectation），我们可以安全地将单次采样的 Reward-to-go 替换为 Q 函数估计值，并使用 V 函数作为 Baseline：
$$ \nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=1}^T \nabla_\theta \log \pi_\theta(a_{i,t}|s_{i,t}) \underbrace{\big({Q}^\pi(s_{i,t}, a_{i,t}) - {V}^\pi(s_{i,t}) \big)}_{A^\pi(s_{i,t}, a_{i,t})} $$

#### 3. 策略评估 (Policy Evaluation)：如何训练 Critic？
我们需要拟合 $\hat{V}_\phi^\pi(s_t)$（参数为 $\phi$ 的神经网络）。有两种方式：
*   **Monte Carlo (MC) 拟合**：让 $\hat{V}_\phi(s_i)$ 逼近真实的整条轨迹采样回报 $\sum r$。
*   **时序差分 (Temporal Difference, TD)**：利用 Bootstrapping（拔靴法），让当前的 V 值逼近单步奖励加上下一步的 V 值。用bias去交换variance。但因此不一定能达到optimal，也不能保证converge。
    $$ \mathcal{L}(\phi) = \frac{1}{2} \sum_i \left\| \hat{V}_\phi^\pi(s_i) - \big( r(s_i, a_i) + \gamma \hat{V}_\phi^\pi(s_{i+1}) \big) \right\|^2 $$
*   使用 TD error 近似优势函数：$A^\pi(s_t, a_t) \approx r(s_t, a_t) + \gamma \hat{V}^\pi_\phi(s_{t+1}) - \hat{V}^\pi_\phi(s_t)$

#### 4. Actor-Critic 工作流
*   **Actor (演员)** $\pi_\theta(a|s)$：负责与环境交互，并根据 Critic 的打分更新自身策略。
*   **Critic (评论家)** $V_\phi(s)$：负责评估当前状态的好坏，并计算 TD error 作为梯度更新的引导。

## 模块六：偏差与方差的终极平衡 (Bias-Variance Tradeoff & GAE)

在估计未来回报时，我们面临一个核心权衡：
*   **Monte Carlo (多步)**：无偏 (No bias)，但高方差 (High variance)。
*   **TD / Actor-Critic (单步)**：低方差 (Lower variance)，但有偏差 (Higher bias，因为神经网络 $V_\phi$ 预估可能是不准确的)。

#### 1. N-step Returns (N步回报)
为了中和两者的缺点，我们可以走 $n$ 步真实轨迹，再用 V 函数估计剩余部分：
$$ G_t^{(n)} = \sum_{t'=t}^{t+n-1} \gamma^{t'-t} r(s_{t'}, a_{t'}) + \gamma^n \hat{V}_\phi(s_{t+n}) $$
这个公式天然鼓励模型去尽早获得reward。

#### 2. GAE (Generalized Advantage Estimation) 广义优势估计
我们不想死板地选择某一个 $n$。GAE 的天才思想是：**将所有 $n$-step 的优势函数进行指数加权平均**。
*   定义单步 TD error： $\delta_{t'} = r_{t'} + \gamma \hat{V}_\phi(s_{t'+1}) - \hat{V}_\phi(s_{t'})$
*   **GAE 公式**：
    $$ \hat{A}_{GAE}^\pi(s_t, a_t) = \sum_{t'=t}^\infty (\gamma \lambda)^{t'-t} \delta_{t'} $$
*   **参数 $\lambda \in [0,1]$ 的作用**：
    *   控制 Monte Carlo 和 TD 之间的混合比例。
    *   $\lambda = 0$ 时，退化为普通单步 Actor-Critic (高偏差，低方差)。
    *   $\lambda = 1$ 时，退化为 Monte Carlo (无偏差，高方差)。
    *   *现代 RL 的标配*：一般设置 $\gamma \approx 0.99, \lambda \approx 0.95$。
